# ⚙️ 03 — Feature Engineering & Outlier Treatment

**Objective:** Rank cities by pollution, identify and handle AQI outliers (especially Ahmedabad), define pollutant thresholds, and prepare data for modeling.

## 3.1 City Rankings — PM2.5 & AQI

In [22]:
city25 = df.groupby('City')['PM2.5'].mean().sort_values(ascending=True)
print(city25.head())
print(city25.tail())

City
Aizawl                16.870354
Ernakulam             24.843210
Thiruvananthapuram    27.824555
Shillong              28.615097
Coimbatore            29.059326
Name: PM2.5, dtype: float64
City
Ahmedabad     66.618467
Lucknow      110.748828
Patna        112.613358
Gurugram     116.331480
Delhi        117.129826
Name: PM2.5, dtype: float64


In [23]:
city25 = df.groupby('City')['AQI'].mean().sort_values(ascending=True)
print(city25.head())
print(city25.tail())

City
Aizawl                34.619469
Shillong              56.316129
Coimbatore            73.191710
Thiruvananthapuram    75.390288
Ernakulam             92.580247
Name: AQI, dtype: float64
City
Lucknow      219.920358
Patna        223.756997
Gurugram     225.050625
Delhi        258.902937
Ahmedabad    432.318815
Name: AQI, dtype: float64


## 3.2 Deep Dive — Ahmedabad

In [24]:
ahmedabad = df[df['City']=='Ahmedabad']
print(ahmedabad[['PM2.5','NO2','SO2','O3','AQI']].mean())

PM2.5     66.618467
NO2       56.512621
SO2       53.201852
O3        38.160179
AQI      432.318815
dtype: float64


In [25]:
print(ahmedabad['AQI'].describe())
ahmedabad.describe()

count    2009.000000
mean      432.318815
std       259.052817
min        48.000000
25%       303.000000
50%       358.000000
75%       492.000000
max      2049.000000
Name: AQI, dtype: float64


,Date,PM2.5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,Year,Month
count,2009,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000,2009.000000
mean,2017-10-01 00:00:00,66.618467,109.052369,20.535453,56.512621,44.511924,20.371573,53.201852,38.160179,4.913668,432.318815,2017.272773,6.249876
min,2015-01-01 00:00:00,3.040000,11.500000,0.060000,0.080000,0.000000,0.060000,0.520000,0.380000,0.000000,48.000000,2015.000000,1.000000
25%,2016-05-17 00:00:00,40.100000,89.375000,10.380000,30.230000,27.840000,10.330000,22.030000,19.620000,1.820000,303.000000,2016.000000,3.000000
50%,2017-10-01 00:00:00,61.090000,109.850000,16.975000,43.320000,38.820000,16.970000,49.945000,43.790000,3.680000,358.000000,2017.000000,6.000000
75%,2019-02-15 00:00:00,78.950000,118.440000,23.750000,74.870000,54.790000,23.750000,67.360000,45.650000,4.720000,492.000000,2019.000000,9.000000
max,2020-07-01 00:00:00,381.690000,586.270000,175.810000,292.020000,246.030000,175.810000,186.080000,162.430000,115.140000,2049.000000,2020.000000,12.000000
std,NaN,34.034664,23.276184,18.268841,40.914369,28.900390,18.420753,34.376713,19.535382,6.958731,259.052817,1.601251,3.439905


## 3.3 Outlier Detection — AQI > 500

In [ ]:
print(df[df['AQI']>500].shape)
print(df[df['AQI']>500]['City'].value_counts())

In [26]:
print(df[df['AQI']>500].groupby('City')['AQI'].describe())
print("----------------------------------------")
print(df[(df['City']== 'Ahmedabad') & (df['AQI']>500)].groupby('Year')['AQI'].count())
print("----------------------------------------")
ahm_median=df[(df['City']== 'Ahmedabad') & (df['AQI']<=500)].groupby('Season')['AQI'].median()
print(ahm_median)

            count        mean         std    min     25%    50%     75%  \
City                                                                      
Ahmedabad   490.0  761.883673  305.992095  502.0  524.25  650.0  900.25   
Amritsar      4.0  708.750000  135.677006  539.0  656.00  713.5  766.25   
Delhi        48.0  559.562500   53.771742  501.0  520.00  537.5  593.00   
Gurugram     22.0  596.954545   91.334799  502.0  531.00  563.0  647.50   
Guwahati      2.0  897.000000   83.438600  838.0  867.50  897.0  926.50   
Hyderabad     4.0  602.500000  115.713727  502.0  508.00  585.5  680.00   
Jorapokhar    7.0  559.428571   32.366944  501.0  549.00  569.0  572.00   
Lucknow      15.0  565.933333   66.340536  503.0  509.50  543.0  594.00   
Patna        25.0  540.720000   31.316290  501.0  520.00  533.0  565.00   
Talcher       4.0  530.000000   27.459060  509.0  514.25  520.5  536.25   

               max  
City                
Ahmedabad   2049.0  
Amritsar     869.0  
Delhi        71

## 3.4 Outlier Capping

**Ahmedabad:** Replace AQI > 500 with season-wise median.
**Other polluted cities:** Cap at 500.

In [27]:
mask = (df['City']=='Ahmedabad') & (df['AQI']>500)
df.loc[mask,'AQI'] = df.loc[mask,'Season'].map(ahm_median)
print(df[(df['City']=='Ahmedabad') & (df['AQI']>500)].shape)


(0, 16)


In [28]:
polluted = ['Delhi', 'Patna', 'Lucknow', 'Gurugram', 'Amritsar']
mask = (df['City'].isin(polluted)) & (df['AQI']>500)
df.loc[mask,'AQI'] = 500 

suspicious = ['Guwahati', 'Jorapokhar','Talcher','Hyderabad']
mask2 = (df['City'].isin(suspicious)) & (df['AQI']>500)
df = df[~mask2]

print(df[df['AQI']>500]['City'].value_counts())


Series([], Name: count, dtype: int64)


## 3.5 City × Season Trend Data

In [29]:
trend = df.groupby(['City','Season'])[['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3', 'Benzene', 'AQI']].mean().reset_index()
print(trend.head(10))


        City   Season      PM2.5        PM10        NO2         CO        SO2  \
0  Ahmedabad   Autumn  96.301705  120.072361  84.545508  29.548557  76.872393   
1  Ahmedabad  Monsoon  42.772231  111.481123  44.495218  18.680016  31.408487   
2  Ahmedabad   Spring  66.781812   90.819475  43.273659  13.488370  52.851721   
3  Ahmedabad   Winter  78.637808  119.124051  69.156517  24.451487  66.789530   
4     Aizawl  Monsoon   3.326129    9.681452   0.223548   0.164194   6.973226   
5     Aizawl   Spring  21.990732   28.330610   0.450854   0.328780   7.531098   
6  Amaravati   Autumn  45.341008   89.752713  28.945271   0.835969  15.916202   
7  Amaravati  Monsoon  17.370909   44.719418  14.930509   0.606800  12.120491   
8  Amaravati   Spring  25.100000   61.715978  10.886522   0.537536  13.275580   
9  Amaravati   Winter  64.475166  111.802435  36.305867   0.618745  16.163100   

          O3    Benzene         AQI  
0  45.678951   7.244295  316.600000  
1  24.610452   3.625195  315.326

In [30]:
top = ['Delhi', 'Patna', 'Gurugram', 'Lucknow', 'Ahmedabad']
bottom = ['Aizawl', 'Ernakulam', 'Thiruvananthapuram', 'Shillong', 'Coimbatore']
selected = top + bottom

trend_filtered = trend[trend['City'].isin(selected)]

## 3.6 City-level Pollutant Statistics

In [31]:
ahm = df[df['City'] == 'Ahmedabad']
print(ahm[['PM2.5','PM10','NO2','CO','SO2','O3','Benzene']].describe())

             PM2.5         PM10          NO2           CO          SO2  \
count  2009.000000  2009.000000  2009.000000  2009.000000  2009.000000   
mean     66.618467   109.052369    56.512621    20.371573    53.201852   
std      34.034664    23.276184    40.914369    18.420753    34.376713   
min       3.040000    11.500000     0.080000     0.060000     0.520000   
25%      40.100000    89.375000    30.230000    10.330000    22.030000   
50%      61.090000   109.850000    43.320000    16.970000    49.945000   
75%      78.950000   118.440000    74.870000    23.750000    67.360000   
max     381.690000   586.270000   292.020000   175.810000   186.080000   

                O3      Benzene  
count  2009.000000  2009.000000  
mean     38.160179     4.913668  
std      19.535382     6.958731  
min       0.380000     0.000000  
25%      19.620000     1.820000  
50%      43.790000     3.680000  
75%      45.650000     4.720000  
max     162.430000   115.140000  


In [33]:
ahm = df[df['City'] == 'Lucknow']
print(ahm[['PM2.5','PM10','NO2','CO','SO2','O3','Benzene']].describe())

             PM2.5          PM10          NO2           CO          SO2  \
count  2009.000000  2.009000e+03  2009.000000  2009.000000  2009.000000   
mean    110.748828  9.251500e+01    33.160854     2.118183     9.889169   
std      78.602457  3.738385e-12    19.762165     2.650864    17.016329   
min      11.390000  9.251500e+01     0.970000     0.000000     1.230000   
25%      48.720000  9.251500e+01    18.090000     1.080000     5.470000   
50%      86.760000  9.251500e+01    28.220000     1.360000     7.500000   
75%     159.210000  9.251500e+01    45.890000     1.970000     9.760000   
max     742.670000  9.251500e+01   121.800000    32.220000   187.020000   

                O3      Benzene  
count  2009.000000  2009.000000  
mean     36.886799     2.823671  
std      19.749207     8.387012  
min       3.410000     0.000000  
25%      23.800000     0.350000  
50%      32.400000     1.350000  
75%      45.800000     2.430000  
max     149.190000   186.430000  


## 3.7 Pollutant Threshold Analysis

In [34]:
pollutants = ['CO', 'NO2', 'SO2', 'Benzene', 'O3']
luck = df[df['City'] == 'Lucknow']

for pol in pollutants:
    print(f"{pol} : max = {luck[pol].max():.2f} , 95th percentile = {luck[pol].quantile(0.95):.2f} ")

CO : max = 32.22 , 95th percentile = 7.44 
NO2 : max = 121.80 , 95th percentile = 72.24 
SO2 : max = 187.02 , 95th percentile = 15.70 
Benzene : max = 186.43 , 95th percentile = 11.30 
O3 : max = 149.19 , 95th percentile = 75.60 


In [35]:
thresholds = {'CO':7.44, 'NO2': 72.24,'SO2': 15.70,'Benzene': 11.30,'O3': 75.60}

In [36]:
for pol, threshold in thresholds.items():
    mask = (df['City'] == 'Ahmedabad') & (df[pol] > threshold)
    med = df[(df['City'] == 'Ahmedabad') & (df[pol] <= threshold)].groupby('Season')[pol].median()
    df.loc[mask,pol] = df.loc[mask,'Season'].map(med)

In [37]:
print(df[df['City'] == 'Ahmedabad'][['CO','NO2','SO2','Benzene','O3']].describe())
print(df[df['City'] == 'Lucknow'][['CO','NO2','SO2','Benzene','O3']].describe())

                CO          NO2          SO2      Benzene           O3
count  2009.000000  2009.000000  2009.000000  2009.000000  2009.000000
mean      4.109311    38.353116    10.175973     3.260513    36.220926
std       1.290232    17.856824     1.561296     2.271171    16.479033
min       0.060000     0.080000     0.520000     0.000000     0.380000
25%       3.320000    29.220000     9.480000     1.820000    19.620000
50%       3.750000    34.820000    10.100000     2.830000    43.090000
75%       4.780000    54.510000    10.580000     4.190000    44.060000
max       7.420000    72.210000    15.670000    11.290000    75.450000
                CO          NO2          SO2      Benzene           O3
count  2009.000000  2009.000000  2009.000000  2009.000000  2009.000000
mean      2.118183    33.160854     9.889169     2.823671    36.886799
std       2.650864    19.762165    17.016329     8.387012    19.749207
min       0.000000     0.970000     1.230000     0.000000     3.410000
25%   